In [38]:
from citibike.citibike_utils import get_trip_duration_mins
from utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [ ]:
from datetime import datetime, timezone

# Use widgets in Databricks Jobs; use local defaults for Databricks Connect notebook runs.
try:
    pipeline_id = dbutils.widgets.get("pipeline_id")
    run_id = dbutils.widgets.get("run_id")
    task_id = dbutils.widgets.get("task_id")
    processed_timestamp = dbutils.widgets.get("processed_timestamp")
    catalog = dbutils.widgets.get("catalog")    
except Exception:
    pipeline_id = "local_notebook"
    run_id = "manual_run"
    task_id = "silver_ingest"
    processed_timestamp = datetime.now(timezone.utc).isoformat()
    catalog = "citibike_dev"

In [40]:
print(f"Pipeline ID: {pipeline_id}")
print(f"Run ID: {run_id}")
print(f"Task ID: {task_id}")
print(f"Processed Timestamp: {processed_timestamp}")
print(f"Catalog: {catalog}")

Pipeline ID: local_notebook
Run ID: manual_run
Task ID: silver_ingest
Processed Timestamp: 2026-05-22T07:32:26.541569+00:00
Catalog: citibike_dev


In [ ]:
df = spark.read.table(f"{catalog}.01_bronze.jc_citibike")
# df = spark.read.table("citibike_dev.01_bronze.jc_citibike")

In [42]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [44]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [46]:
df = df.withColumn("metadata", 
              create_map(
                  lit("pipeline_id"), lit(pipeline_id),
                  lit("run_id"), lit(run_id),
                  lit("task_id"), lit(task_id),
                  lit("processed_timestamp"), lit(processed_timestamp)
                  ))

In [47]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
    )

In [ ]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.02_silver.jc_citibike")